# XLM-RoBERTa Fine-Tuning — Emotion Classification
**Project:** NLP-Based Emotion-Aware Journal Analysis System  
**Model:** xlm-roberta-base  
**Dataset:** final_multilingual_dataset.csv (84,380 rows · 4 languages · 11 emotion classes)  
**Works on:** MacBook MPS · Google Colab (T4) · NVIDIA GPU · CPU fallback

## 1. Install Dependencies

In [13]:
# Run this cell first — installs everything needed
# On Colab: runs normally
# On MacBook: run once, then restart kernel

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "transformers", "torch", "scikit-learn", "pandas", "numpy",
    "tqdm", "accelerate", "-q"
])
print("Dependencies installed.")

Dependencies installed.


In [14]:
import torch
import os

import pandas as pd
import numpy as np
from ast import literal_eval

from sklearn.model_selection import train_test_split

from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

import torch.nn as nn
from transformers import AutoModel

from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import numpy as np
import torch

from sklearn.metrics import f1_score, classification_report

import os, json
from tqdm import tqdm
from sklearn.metrics import f1_score


## 2. Setup — Device Detection & Config

In [15]:
import torch
import os

# ── Auto device detection ──────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    BATCH_SIZE = 32        # Colab T4 can handle 32
    print(f"CUDA GPU detected: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    BATCH_SIZE = 16        # MacBook MPS — keep lower
    print("Apple MPS detected (MacBook GPU)")
else:
    DEVICE = torch.device("cpu")
    BATCH_SIZE = 8
    print("CPU only — training will be slow")

print(f"Device: {DEVICE} | Batch size: {BATCH_SIZE}")

# ── Config ─────────────────────────────────────────────────────────────────
MODEL_NAME      = "xlm-roberta-base"
MAX_LEN         = 128          # 128 is enough for journal entries; saves memory
EPOCHS          = 5
LR              = 2e-5
THRESHOLD       = 0.40         # Emotion flagged active if score > 0.40
SAVE_DIR        = "./model_output"
DATASET_PATH    = "final_multilingual_dataset.csv"   # update path if needed

TARGET_EMOTIONS = [
    "joy", "trust", "fear", "surprise", "sadness",
    "disgust", "anger", "anticipation", "love", "optimism", "pessimism"
]
NUM_LABELS = len(TARGET_EMOTIONS)

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Config ready. Labels: {TARGET_EMOTIONS}")

Apple MPS detected (MacBook GPU)
Device: mps | Batch size: 16
Config ready. Labels: ['joy', 'trust', 'fear', 'surprise', 'sadness', 'disgust', 'anger', 'anticipation', 'love', 'optimism', 'pessimism']


## 3. Load & Parse Dataset

In [16]:
import pandas as pd
import numpy as np
from ast import literal_eval

DATASET_PATH = 'final_multilingual_dataset.csv'
df = pd.read_csv(DATASET_PATH)
print(f"Loaded: {df.shape[0]} rows")

# ── Parse label_vector from string → numpy array ───────────────────────────
# Stored as "[0. 1. 0. ...]" string in CSV — need to convert back
def parse_label_vector(v):
    v = str(v).strip().replace('[', '').replace(']', '')
    return np.array([float(x) for x in v.split()], dtype=np.float32)

df["label_vector"] = df["label_vector"].apply(parse_label_vector)

# ── Sanity check ───────────────────────────────────────────────────────────
assert df["label_vector"].iloc[0].shape[0] == NUM_LABELS, "Label dimension mismatch!"
assert df["text"].isnull().sum() == 0, "Null texts found!"

print(f"Label vector shape: {df['label_vector'].iloc[0].shape}")
print(f"Sample text: {df['text'].iloc[0][:80]}")
print(f"Sample label: {df['label_vector'].iloc[0]}")

# ── Class distribution ─────────────────────────────────────────────────────
label_matrix = np.stack(df["label_vector"].values)
counts = label_matrix.sum(axis=0).astype(int)
print("\nClass distribution:")
for e, c in zip(TARGET_EMOTIONS, counts):
    print(f"  {e:<15} {c:>6}")

Loaded: 84380 rows
Label vector shape: (11,)
Sample text: yes it is i’m 99.9% sure! thank you! enjoy your gold.
Sample label: [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Class distribution:
  joy              14436
  trust            11928
  fear              2464
  surprise         11556
  sadness           6240
  disgust           2704
  anger            18652
  anticipation      9616
  love              9664
  optimism          5308
  pessimism         4440


## 4. Train / Validation / Test Split

In [17]:
from sklearn.model_selection import train_test_split

# 80 / 10 / 10 split
train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 67504 | Val: 8438 | Test: 8438


## 5. Tokenizer & Dataset Class

In [18]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")

class EmotionDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.texts  = dataframe["text"].tolist()
        self.labels = dataframe["label_vector"].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.float32)
        }

train_dataset = EmotionDataset(train_df, tokenizer, MAX_LEN)
val_dataset   = EmotionDataset(val_df,   tokenizer, MAX_LEN)
test_dataset  = EmotionDataset(test_df,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

/Users/tejaramidi/anaconda3/envs/emotion_nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Tokenizer loaded: xlm-roberta-base
Train batches: 4219 | Val batches: 528


## 6. Model Definition

In [19]:
import torch.nn as nn
from transformers import AutoModel

class EmotionClassifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        self.encoder  = AutoModel.from_pretrained(model_name)
        self.dropout  = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_labels)
        # Sigmoid applied at inference — not here (BCEWithLogitsLoss handles it)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]   # [CLS] token
        cls_output = self.dropout(cls_output)
        logits     = self.classifier(cls_output)           # shape: (batch, 11)
        return logits

model = EmotionClassifier(MODEL_NAME, NUM_LABELS).to(DEVICE)
print(f"Model loaded and moved to {DEVICE}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Model loaded and moved to mps
Trainable parameters: 278,052,107


## 7. Loss, Optimizer & Scheduler

**Weighted BCE** handles the class imbalance (anger: 18K vs fear: 2.4K).  
Weight for each class = `total_samples / (num_classes × class_count)` — minority classes get higher weight.

In [20]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import numpy as np
import torch

NUM_EPOCHS = 5

# ── Correct pos_weight: neg_count / pos_count per class ───────────────────
# This is what BCEWithLogitsLoss expects for pos_weight
# High ratio = minority class gets more penalty when missed
labels_arr = np.stack(train_df["label_vector"].values)
pos_counts = labels_arr.sum(axis=0)
neg_counts = len(labels_arr) - pos_counts
pos_weight  = torch.tensor(neg_counts / pos_counts, dtype=torch.float32).to(DEVICE)

print("pos_weight per class:")
for e, w in zip(TARGET_EMOTIONS, pos_weight.cpu().numpy()):
    print(f"  {e:<15} {w:.2f}x")

# ── Loss ──────────────────────────────────────────────────────────────────
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# ── Optimizer — 2e-5 is standard for XLM-RoBERTa fine-tuning ─────────────
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

# ── Scheduler — must use same NUM_EPOCHS as training loop ─────────────────
total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler    = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"\nTotal steps: {total_steps} | Warmup: {warmup_steps} | LR: 2e-5")

pos_weight per class:
  joy             4.86x
  trust           6.07x
  fear            33.21x
  surprise        6.34x
  sadness         12.60x
  disgust         30.38x
  anger           3.49x
  anticipation    7.82x
  love            7.79x
  optimism        14.86x
  pessimism       18.03x

Total steps: 21095 | Warmup: 2109 | LR: 2e-5


## 8. Evaluation Metrics

In [11]:
from sklearn.metrics import f1_score, classification_report

def evaluate(model, loader, threshold=THRESHOLD):
    model.eval()
    all_preds  = []
    all_labels = []
    total_loss = 0.0

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)

            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels)
            total_loss += loss.item()

            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= threshold).astype(int)

            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())

    all_preds  = np.vstack(all_preds)
    all_labels = np.vstack(all_labels).astype(int)

    micro_f1 = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    avg_loss = total_loss / len(loader)

    return avg_loss, micro_f1, macro_f1, all_preds, all_labels

print("Evaluation function ready.")

Evaluation function ready.


In [21]:
import numpy as np
import torch

labels = np.array(train_df["label_vector"].tolist())

pos_counts = labels.sum(axis=0)
neg_counts = len(labels) - pos_counts

pos_weight = torch.tensor(neg_counts / pos_counts, dtype=torch.float).to(DEVICE)

print("pos_weight:", pos_weight)

pos_weight: tensor([ 4.8577,  6.0714, 33.2139,  6.3430, 12.6042, 30.3826,  3.4922,  7.8194,
         7.7942, 14.8646, 18.0313], device='mps:0')


## 9. Training Loop

Saves the **best checkpoint** based on validation Micro-F1.  
Early stopping if no improvement for 2 consecutive epochs.

In [ ]:
import os, json
from tqdm import tqdm
from sklearn.metrics import f1_score

CHECKPOINT_PATH = "last_text_checkpoint.pth"
BEST_MODEL_PATH = "best_text_model.pth"
STATE_FILE      = "training_state.json"
PATIENCE        = 2

# ── Resume detection ───────────────────────────────────────────────────────
if os.path.exists(CHECKPOINT_PATH):
    print("Resuming from checkpoint...")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    start_epoch      = ckpt["epoch"] + 1
    best_val_f1      = ckpt["best_val_f1"]
    patience_counter = ckpt["patience_counter"]
    print(f"Resumed from epoch {start_epoch} | Best F1 so far: {best_val_f1:.4f}")
else:
    start_epoch      = 0
    best_val_f1      = 0.0
    patience_counter = 0
    print("Starting fresh training")

history = []

# ── Training loop ──────────────────────────────────────────────────────────
for epoch in range(start_epoch, NUM_EPOCHS):
    print(f"\n===== Epoch {epoch+1}/{NUM_EPOCHS} =====")

    # TRAIN
    model.train()
    total_train_loss = 0

    for batch in tqdm(train_loader):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()   # ← THIS WAS MISSING — LR never changed before

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    print(f"Train Loss: {avg_train_loss:.4f}")

    # VALIDATE — compute both loss AND F1
    model.eval()
    total_val_loss = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)

            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels)
            total_val_loss += loss.item()

            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= THRESHOLD).astype(int)
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy().astype(int))

    avg_val_loss = total_val_loss / len(val_loader)
    all_preds    = np.vstack(all_preds)
    all_labels   = np.vstack(all_labels)
    val_micro_f1 = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    val_macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    print(f"Val Loss:     {avg_val_loss:.4f}")
    print(f"Val Micro-F1: {val_micro_f1:.4f} | Val Macro-F1: {val_macro_f1:.4f}")

    history.append({
        "epoch": epoch+1,
        "train_loss": round(avg_train_loss, 4),
        "val_loss": round(avg_val_loss, 4),
        "val_micro_f1": round(val_micro_f1, 4),
        "val_macro_f1": round(val_macro_f1, 4)
    })

    # Save checkpoint after every epoch (for resume)
    torch.save({
        "epoch":              epoch,
        "model_state_dict":   model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),  # ← new
        "best_val_f1":        best_val_f1,
        "patience_counter":   patience_counter
    }, CHECKPOINT_PATH)
    print("Checkpoint saved")

    # Save best model by Micro-F1 (not loss)
    if val_micro_f1 > best_val_f1:
        best_val_f1      = val_micro_f1
        patience_counter = 0
        torch.save({"model_state_dict": model.state_dict()}, BEST_MODEL_PATH)
        print(f"Best model saved! (Micro-F1: {best_val_f1:.4f})")
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{PATIENCE}")
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

print(f"\nTraining complete. Best Val Micro-F1: {best_val_f1:.4f}")
import pandas as pd
print(pd.DataFrame(history).to_string(index=False))

Resuming from checkpoint...
Resumed from epoch 3 | Best F1 so far: 0.5256

===== Epoch 4/5 =====


100%|██████████| 4219/4219 [1:01:26<00:00,  1.14it/s]


Train Loss: 0.4421
Val Loss:     0.6281
Val Micro-F1: 0.5963 | Val Macro-F1: 0.5738
Checkpoint saved
Best model saved! (Micro-F1: 0.5963)

===== Epoch 5/5 =====


100%|██████████| 4219/4219 [1:03:33<00:00,  1.11it/s]


Train Loss: 0.3737
Val Loss:     0.6343
Val Micro-F1: 0.6145 | Val Macro-F1: 0.5921
Checkpoint saved
Best model saved! (Micro-F1: 0.6145)

Training complete. Best Val Micro-F1: 0.6145
 epoch  train_loss  val_loss  val_micro_f1  val_macro_f1
     4      0.4421    0.6281        0.5963        0.5738
     5      0.3737    0.6343        0.6145        0.5921


In [22]:
checkpoint = torch.load("best_model.pth", map_location=DEVICE)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)

model.eval()

print("Best model loaded")

Best model loaded


In [23]:
# from google.colab import drive
# drive.mount('/content/drive')

In [24]:
# import shutil

# shutil.copy('/content/best_model.pth', '/content/drive/MyDrive/')
# shutil.copy('/content/last_checkpoint.pth', '/content/drive/MyDrive/')

## 10. Test Set Evaluation

In [25]:
from sklearn.metrics import f1_score, accuracy_score

model.eval()

all_preds = []
all_labels = []

threshold = 0.5

with torch.no_grad():
    for batch in test_loader:

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs

        # 🔥 IMPORTANT FIX
        probs = torch.sigmoid(logits)

        preds = (probs > threshold).int()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Convert to numpy
import numpy as np
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Metrics
f1 = f1_score(all_labels, all_preds, average="macro")
acc = accuracy_score(all_labels, all_preds)

print(f"F1 Score: {f1:.4f}")
print(f"Accuracy: {acc:.4f}")

F1 Score: 0.6137
Accuracy: 0.3584


In [28]:
from sklearn.metrics import f1_score
import numpy as np
import torch

# ── Recompute everything fresh from test_loader ────────────────────────────
model.eval()
all_probs  = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        logits = model(input_ids, attention_mask)
        probs  = torch.sigmoid(logits).cpu().numpy()

        all_probs.append(probs)
        all_labels.append(labels.cpu().numpy().astype(int))

all_probs  = np.vstack(all_probs)
all_labels = np.vstack(all_labels)

print(f"all_probs shape:  {all_probs.shape}")   # should be (8438, 11)
print(f"all_labels shape: {all_labels.shape}")  # should be (8438, 11)

# ── Threshold tuning ───────────────────────────────────────────────────────
print("\nThreshold tuning results:")
print(f"{'Threshold':<12} {'Micro-F1':<12} {'Macro-F1'}")
print("-" * 38)
for thresh in [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55]:
    preds    = (all_probs >= thresh).astype(int)
    micro_f1 = f1_score(all_labels, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(all_labels, preds, average="macro", zero_division=0)
    print(f"{thresh:<12} {micro_f1:<12.4f} {macro_f1:.4f}")

all_probs shape:  (8438, 11)
all_labels shape: (8438, 11)

Threshold tuning results:
Threshold    Micro-F1     Macro-F1
--------------------------------------
0.25         0.5677       0.5470
0.3          0.5863       0.5647
0.35         0.6021       0.5797
0.4          0.6150       0.5925
0.45         0.6253       0.6025
0.5          0.6364       0.6137
0.55         0.6467       0.6244


In [29]:
print("\nExtended threshold tuning:")
print(f"{'Threshold':<12} {'Micro-F1':<12} {'Macro-F1'}")
print("-" * 38)
for thresh in [0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]:
    preds    = (all_probs >= thresh).astype(int)
    micro_f1 = f1_score(all_labels, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(all_labels, preds, average="macro", zero_division=0)
    
    # Also check how many predictions are all-zero (no emotion detected)
    all_zero = (preds.sum(axis=1) == 0).sum()
    
    print(f"{thresh:<12} {micro_f1:<12.4f} {macro_f1:<12.4f}  all-zero: {all_zero}")


Extended threshold tuning:
Threshold    Micro-F1     Macro-F1
--------------------------------------
0.55         0.6467       0.6244        all-zero: 3
0.6          0.6524       0.6313        all-zero: 8
0.65         0.6608       0.6398        all-zero: 27
0.7          0.6690       0.6494        all-zero: 66
0.75         0.6731       0.6543        all-zero: 134
0.8          0.6725       0.6552        all-zero: 259
0.85         0.6690       0.6542        all-zero: 490
0.9          0.6596       0.6499        all-zero: 989


## 12. Quick Inference Test

Test the model on a few sample journal entries.

In [30]:
def predict_emotions(text):

    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    input_ids = inputs["input_ids"].to(DEVICE)
    attention_mask = inputs["attention_mask"].to(DEVICE)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    logits = outputs

    probs = torch.sigmoid(logits).cpu().numpy()[0]

    threshold = 0.75   # was 0.40
    preds = (probs > threshold).astype(int)

    return preds, probs

In [31]:
test_sentence = "I feel very happy today but also a little nervous"

preds, probs = predict_emotions(test_sentence)

print("Sentence:", test_sentence)
print("Predictions:", preds)
print("Probabilities:", probs)

Sentence: I feel very happy today but also a little nervous
Predictions: [1 0 1 0 0 0 0 0 0 0 0]
Probabilities: [0.9460569  0.02370986 0.9371917  0.14962138 0.00829878 0.01042607
 0.02494476 0.07001944 0.04020923 0.01904065 0.01348412]


In [32]:
for i in range(5):
    print(df["text"].iloc[i])
    print(df["label_vector"].iloc[i])
    print()

yes it is i’m 99.9% sure! thank you! enjoy your gold.
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

oh you got me. good one.
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

you got it, just keep doing whatever you were doing the past 20-whatever days. constrict that demon, dude.
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

> i'll only debate subjects i'm knowledgable of. then you're not good at debating.
[0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0.]

thank you, i need it. i am currently trying to learn french, would that help ?
[0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0.]



In [35]:
# Test with journal-style entries
test_journals = [
    # English
    "Today was really exhausting. I had three back to back meetings and couldn't finish any of my actual work. I feel like I am running behind on everything and it is stressing me out.",
    
    # English - positive
    "Finally submitted my project today. I have been working on it for weeks and I am so relieved it is done. Feeling proud of myself honestly.",
    
    # Telugu
    "ఈరోజు చాలా అలసిపోయాను. పని ఒత్తిడి వల్ల నిద్ర కూడా సరిగా పడటం లేదు. ఏం చేయాలో అర్థం కావడం లేదు.",
    
    # Hindi
    "आज बहुत अच्छा दिन था। दोस्तों के साथ समय बिताया और बहुत मज़ा आया। मन बहुत हल्का लग रहा है।",
]

model.eval()
print("=" * 60)
for journal in test_journals:
    encoding = tokenizer(
        journal,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    input_ids      = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs  = torch.sigmoid(logits).cpu().numpy()[0]

    active = [(TARGET_EMOTIONS[i], round(float(probs[i]), 3))
              for i in range(len(TARGET_EMOTIONS)) if probs[i] >= 0.75]
    dominant = TARGET_EMOTIONS[int(probs.argmax())]

    print(f"Journal: {journal[:70]}...")
    print(f"Dominant : {dominant}")
    print(f"Active   : {active if active else 'None above threshold'}")
    print(f"All scores: { {e: round(float(p),2) for e,p in zip(TARGET_EMOTIONS, probs)} }")
    print("-" * 60)

Journal: Today was really exhausting. I had three back to back meetings and cou...
Dominant : pessimism
Active   : [('sadness', 0.963), ('pessimism', 0.978)]
All scores: {'joy': 0.01, 'trust': 0.01, 'fear': 0.39, 'surprise': 0.08, 'sadness': 0.96, 'disgust': 0.02, 'anger': 0.2, 'anticipation': 0.01, 'love': 0.02, 'optimism': 0.01, 'pessimism': 0.98}
------------------------------------------------------------
Journal: Finally submitted my project today. I have been working on it for week...
Dominant : joy
Active   : [('joy', 0.989)]
All scores: {'joy': 0.99, 'trust': 0.53, 'fear': 0.0, 'surprise': 0.05, 'sadness': 0.01, 'disgust': 0.01, 'anger': 0.02, 'anticipation': 0.01, 'love': 0.02, 'optimism': 0.03, 'pessimism': 0.0}
------------------------------------------------------------
Journal: ఈరోజు చాలా అలసిపోయాను. పని ఒత్తిడి వల్ల నిద్ర కూడా సరిగా పడటం లేదు. ఏం...
Dominant : surprise
Active   : [('fear', 0.91), ('surprise', 0.971)]
All scores: {'joy': 0.04, 'trust': 0.01, 'fear': 0.91, 